# Aufgabe

Macht es Sinn die Spalte "fuelType" und "gearBox" mit in das Modell einfließen zu lassen? Können wir damit unser Modell verbessern (Bestimmtheitsmaß)?


In [3]:
import pandas as pd

df = pd.read_csv("./data/Autos/autos.csv.bz2", encoding="ISO-8859-1")

df = df[df["offerType"] == "Angebot"]
df = df[df["vehicleType"] == "kleinwagen"]
df = df[df["notRepairedDamage"] == "nein"]
df = df[(df["fuelType"] == "benzin") | (df["fuelType"] == "diesel") | (df["fuelType"] == "hybrid")]

df.dropna(inplace=True)

df.head()

,dateCrawled,name,seller,offerType,price,abtest,vehicleType,yearOfRegistration,gearbox,powerPS,model,kilometer,monthOfRegistration,fuelType,brand,notRepairedDamage,dateCreated,nrOfPictures,postalCode,lastSeen
3,2016-03-17 16:54:04,GOLF_4_1_4__3TÜRER,privat,Angebot,1500,test,kleinwagen,2001,manuell,75,golf,150000,6,benzin,volkswagen,nein,2016-03-17 00:00:00,0,91074,2016-03-17 17:40:17
4,2016-03-31 17:25:20,Skoda_Fabia_1.4_TDI_PD_Classic,privat,Angebot,3600,test,kleinwagen,2008,manuell,69,fabia,90000,7,diesel,skoda,nein,2016-03-31 00:00:00,0,60437,2016-04-06 10:17:21
17,2016-03-20 10:25:19,Renault_Twingo_1.2_16V_Aut.,privat,Angebot,1750,control,kleinwagen,2004,automatik,75,twingo,150000,2,benzin,renault,nein,2016-03-20 00:00:00,0,65599,2016-04-06 13:16:07
23,2016-03-12 19:43:07,Stadtflitzer,privat,Angebot,450,test,kleinwagen,1997,manuell,50,arosa,150000,5,benzin,seat,nein,2016-03-12 00:00:00,0,9526,2016-03-21 01:46:11
29,2016-03-08 19:55:19,Fiat_Punto_1.2,privat,Angebot,690,test,kleinwagen,2003,manuell,60,punto,150000,3,benzin,fiat,nein,2016-03-08 00:00:00,0,86199,2016-03-09 11:45:28


In [4]:
df["fuelType"].unique()

array(['benzin', 'diesel', 'hybrid'], dtype=object)

## Model Score Basis

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X = df[["yearOfRegistration", "kilometer"]]
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

print(model.score(X_test, y_test))
print(model.score(X_train, y_train))


0.5523617826454987
0.5519447479395423


## Model Score mit Gearbox und fuelType im Modell 

### Nur Gearbox

In [6]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

X = df[["yearOfRegistration", "kilometer", "brand", "gearbox"]]
ct = ColumnTransformer(
    [("gearbox", OneHotEncoder(drop = "first"), ["gearbox"]),
     ("brand", OneHotEncoder(drop = "first"), ["brand"])],
    remainder="passthrough"
)
X.head()

ct.fit(X)
X_transformed = ct.transform(X)

In [7]:
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

print(model.score(X_test, y_test))
print(model.score(X_train, y_train))

0.6238783986709888
0.6703549275095053


### Nur fuelType

In [8]:
X = df[["yearOfRegistration", "kilometer", "brand", "fuelType"]]
ct = ColumnTransformer(
    [("fuelType", OneHotEncoder(drop = "first"), ["fuelType"]),
     ("brand", OneHotEncoder(drop = "first"), ["brand"])],
    remainder="passthrough"
)
X.head()

ct.fit(X)
X_transformed = ct.transform(X)

In [9]:
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

print(model.score(X_test, y_test))
print(model.score(X_train, y_train))

0.6610365751911049
0.6591202435947148


### fuelType + gearBox

In [10]:
X = df[["yearOfRegistration", "kilometer", "brand", "fuelType", "gearbox"]]
ct = ColumnTransformer(
    [("fuelType", OneHotEncoder(drop = "first"), ["fuelType"]),
     ("brand", OneHotEncoder(drop = "first"), ["brand"]),
     ("gearbox", OneHotEncoder(drop = "first"), ["gearbox"])],
    remainder="passthrough"
)
X.head()

ct.fit(X)
X_transformed = ct.transform(X)

In [11]:
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

print(model.score(X_test, y_test))
print(model.score(X_train, y_train))

0.655490120487241
0.6628703468488684


In [12]:
scores = []

for i in range(0, 1000):
    X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.2)

    model = LinearRegression()
    model.fit(X_train, y_train)

    scores.append(model.score(X_test, y_test))
    
scores.sort(reverse=True)
print(scores)

[0.7017369517115343, 0.7015126036652906, 0.6990947877567746, 0.6978518266200014, 0.6957279558724598, 0.69476973256779, 0.6945121401068612, 0.6943683599749539, 0.6942377776166216, 0.6940755642856247, 0.693973910124702, 0.6937328420117752, 0.6932428285982704, 0.6928586644203589, 0.6924500186357763, 0.6924251495564915, 0.6921621465603319, 0.6921287804640328, 0.6920879048324033, 0.69203469923916, 0.6917599475780549, 0.6913519792382017, 0.6911158613588448, 0.6910264632219167, 0.6908474898826386, 0.6905794299686969, 0.6905420006068123, 0.6904074965118578, 0.6904047044893618, 0.6903608047105463, 0.6900915184698487, 0.6898664657380138, 0.6897012394349482, 0.6896015531554596, 0.6893788304665189, 0.6893734101638098, 0.6893323109434406, 0.6891048118542078, 0.6890923911739305, 0.6889822221250675, 0.6889768395449989, 0.6887638652828223, 0.6885357573526549, 0.6883919580450579, 0.6883906285161512, 0.6882881674933448, 0.6882787763175929, 0.6882294631857759, 0.687908295956813, 0.6879042693192883, 0.687

In [13]:
import numpy as np

print(np.mean(scores))

0.6624946429512811


## Vorhersage

In [14]:
import pandas as pd

X_pred = pd.DataFrame([
    [150000, 2000, "bmw", "automatik", "benzin"]
], columns=["kilometer", "yearOfRegistration", "brand", "gearbox", "fuelType"])

model.predict(ct.transform(X_pred))

array([2299.47384466])